# Libs 

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from numpy.typing import NDArray
from skimage import io
from tqdm.auto import tqdm

## A GPU está sendo utilizada? se sim, qual?

In [3]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 4060 Ti


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Paths, seeds e paletas

In [5]:
IMAGES_PATH = Path("./dados/10x/Images")
MASKS_PATH = Path("./dados/10x/Masks")
SEED_1 = 9
SEED_2 = 23
SEED_3 = 69
TRAIN_LIST = "./dados/train_files.txt"
TEST_LIST = "./dados/test_files.txt"
VALIDATION_LIST = "./dados/validation_files.txt"
PALETA = np.array([
    [0, 0, 0],        # 0 BKG fundo
    [248, 123, 168],  # 1 KER queratina
    [73, 0, 106],     # 2 EPI epiderme
    [236, 85, 157],   # 3 PAP derme papilar
    [181, 9, 130],    # 4 RET derme reticular
    [254, 246, 242],  # 5 HYP hipoderme
    [108, 0, 115],    # 6 GLD glândulas
    [216, 47, 148],   # 7 FOL folículos
    [145, 1, 122],    # 8 INF inflamação
    [127, 255, 255],  # 9 BCC
    [127, 255, 142],  # 10 SCC
    [255, 127, 127],  # 11 IEC
])

## Utils

In [6]:
def read_list(path: str) -> list[str]:
    
    file_path = Path(path)
    
    file_list = file_path.read_text().splitlines()
    
    file_list_stripped = [row.strip() for row in file_list if row.strip() != ""]
    
    print(file_list_stripped[0:4])
    print(len(file_list_stripped))
    
    return file_list_stripped

In [7]:
def calculate_euclidean_distance(mask: NDArray[np.int32], paleta: NDArray) -> NDArray[np.float64]:
    
    calculated_distances = [((mask - color) ** 2).sum(axis= -1) for color in paleta]
    
    distances = np.stack(
        calculated_distances, 
        axis= -1,
    )
    
    sqrt_distances = np.sqrt(distances)
    
    return sqrt_distances
    

In [8]:
def masks_for_classes(mask_rgb: NDArray[np.uint8]) -> NDArray[np.uint8]:
    
    mask = mask_rgb.astype(np.int32)
    
    sqrt_distances = calculate_euclidean_distance(mask, PALETA)
    
    closest_classes = sqrt_distances.argmin(axis=-1).astype(np.uint8)

    return closest_classes

In [9]:
def load_images_list(images_names_list: list[str]) -> list[tuple[NDArray[np.uint8], NDArray[np.uint8]]]:
    
    images_list = []
    masks_list = []
    
    for name in tqdm(images_names_list, desc="Loading images"):
        
        image = io.imread(IMAGES_PATH/f"{name}.tif")
        images_list.append(image)
        
        mask = io.imread(MASKS_PATH/f"{name}.png")
        masks_list.append(masks_for_classes(mask))
        
    paired_lists = list(zip(images_list, masks_list))    
    
    return paired_lists

## Data

In [10]:
train_list = read_list(TRAIN_LIST)

['SCC_33', 'BCC_101', 'BCC_122', 'IEC_15']
231


In [11]:
test_list = read_list(TEST_LIST)

['BCC_10', 'BCC_116', 'SCC_4', 'SCC_42']
31


In [12]:
validation_list = read_list(VALIDATION_LIST)

['IEC_32', 'SCC_12', 'IEC_61', 'IEC_18']
28


In [19]:
train = load_images_list(train_list)
test = load_images_list(test_list)
validation = load_images_list(validation_list)


Loading images:   0%|          | 0/231 [00:00<?, ?it/s]

Loading images:   0%|          | 0/31 [00:00<?, ?it/s]

Loading images:   0%|          | 0/28 [00:00<?, ?it/s]